In [1]:
from jobflow_remote import submit_flow, set_run_config
from autoplex.auto.GenMLFF.jobs import initial_iteration

/leonardo_work/EUHPC_A04_113/Alberto/GenMLFF-progect/.env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/leonardo_work/EUHPC_A04_113/Alberto/GenMLFF-progect/.env/lib/python3.11/site-packages/nequip/__init__.py:20: UserWarning: !! PyTorch version 2.2.1+cu121 found. Upstream issues in PyTorch versions 1.13.* and 2.* have been seen to cause unusual performance degredations on some CUDA systems that become worse over time; see https://github.com/mir-group/nequip/discussions/311. The best tested PyTorch version to use with CUDA devices is 1.11; while using other versions if you observe this problem, an unexpected lack of this problem, or other strange behavior, please post in the linked GitHub issue.
  warnings.warn(


In [2]:
#Define RSS test parameters
rss_test_params = {
    "tag": "SiO2", #Tag of systems. It can also be used for setting up elements and stoichiometry.
    "generated_struct_numbers": [10, 5], #Expected number of generated randomized unit cells for each run
    "buildcell_options": [ #Buildcell params for each buildcell run
        {"NFORM": '1', "SYMMOPS": "1-4", "SLACK": 0.25, "OVERLAP" : 0.1, "NATOM" : '{6,8,10,12,14,16,18,20,22,24}'}, 
        {"NFORM": '1', "SYMMOPS": "1-4", "SLACK": 0.25, "OVERLAP" : 0.1, "NATOM" : '{7,9,11,13,15,17,19,21,23}'}
    ],
    "fragment_file": None, #Fragment(s) for random structures, e.g. molecules, to be placed indivudally intact.
    "remove_tmp_files": True, #Remove all temporary files raised by buildcell to save memory
    "num_processes": 1, #Number of processes to use for parallel computation
}

In [3]:
#Define MLIP-scf test parameters
mlscf_test_params = {
    "mlip_type": "MACE", #Type of MLIP to be used
    "mlip_kwargs": {
        "model_paths":"/leonardo_work/EUHPC_A04_113/Alberto/mace/pre-trained-models/mace-mpa-0-medium.model", 
        "device" : "cuda"
    },
    "dimer" : True
}

In [4]:
#Define ensemble dataset split parameters
ensemble_split_params = {
        "num_models": 4,
        "test_ratio": 0.05,
        "distill_force_max": 20.0, #eV/Å
    }

In [ ]:
#Define parameters for ensemble training
ensemble_train_params = {
    "num_models": 4,
    "remove_model_datasets": True,    
    "mlip_type": "MACE",
    "mlip_train_kwargs": {
        "device": "cuda",
        "batch_size": 2,
        "max_num_epochs": 10,
        "swa": False,
        "E0s": "{8 : -561.5463794436, 14 : -1113.927588872}",
        "enable_cueq": False,
    },
}

In [6]:
# #Define MLIPlabelling test parameters
# mlip_type = "MACE"
# mace_kwargs={
#     "model_paths":"/leonardo_work/EUHPC_A04_113/Alberto/mace/pre-trained-models/mace-mpa-0-medium.model", 
#     "device" : "cuda"
#     }

In [7]:
#Define resources
parallel_cpu_resources = {
    "account": "IscrB_MLSilDia",
    "partition": "boost_usr_prod",
    "qos": "boost_qos_dbg",
    "time": "00:30:00",
    "nodes": 1,
    "ntasks_per_node": 32,
    "cpus_per_task": 1,
    "gres": "gpu:0",
    "mem": "480000",
    "job_name": "mlff_relax",
    "qerr_path": "mlff_relax.err",
    "qout_path": "mlff_relax.out",
}

serial_cpu_resources = {
    "account": "IscrB_MLSilDia",
    "partition": "boost_usr_prod",
    "qos": "boost_qos_dbg",
    "time": "00:30:00",
    "nodes": 1,
    "ntasks_per_node": 1,
    "cpus_per_task": 32,
    "gres": "gpu:0",
    "mem": "480000",
    "job_name": "mlff_relax",
    "qerr_path": "mlff_relax.err",
    "qout_path": "mlff_relax.out",
}

serial_gpu_resources = {
    "account": "IscrB_MLSilDia", 
    "partition": "boost_usr_prod",
    "qos": "boost_qos_dbg",
    "time": "00:30:00",
    "nodes": 1,
    "ntasks_per_node": 1,
    "cpus_per_task": 8,
    "gres": "gpu:1",
    "mem": "120000",
    "job_name": "mlff_relax",
    "qerr_path": "mlff_relax.err",
    "qout_path": "mlff_relax.out",
    }

In [8]:
# #Define RSS test flow
# rss_job_test = RSS(**rss_test_params)

# #Define MLIP labelling test flow
# mlscf_job_test = MLscf(
#     mlip_type=mlip_type,
#     mlip_kwargs=mace_kwargs,
#     structure_paths=rss_job_test.output,
#     dimer=True,
# )

# #Define flow
# rss_mlip_flow = Flow([rss_job_test, mlscf_job_test], name="rss_mlip_flow")

In [9]:
# Start the initial iteration
init_job = initial_iteration(
    rss_params=rss_test_params,
    mlip_params=mlscf_test_params,
    dataset_params=ensemble_split_params,
    train_params=ensemble_train_params,
)

In [10]:
init_job = set_run_config(
    init_job, name_filter="RSS", worker="mlff_relax_local", exec_config="rss_config", resources=serial_cpu_resources
)

init_job = set_run_config(
    init_job, name_filter="MLscf", worker="mlff_relax_local", exec_config="rss_config", resources=serial_gpu_resources
)

init_job = set_run_config(
    init_job, name_filter="training_mlip", worker="mlff_relax_local", exec_config="rss_config", resources=serial_gpu_resources
)

In [11]:
# rss_job_test = set_run_config(
#     rss_flow_test, name_filter="init_RSS", exec_config="rss_config", dynamic=False
# )
# rss_flow_test = set_run_config(
#     rss_flow_test, name_filter="do_randomized_structure_generation", worker="mlff_relax_local", exec_config="rss_config", resources=serial_cpu_resources
# )

In [12]:
# Append RSSautoplex-flow to jf jobs
submit_flow(
    init_job, worker="local_worker",
    resources={}, 
    project="GenMLFF",
)

2025-05-13 00:29:00,572 - INFO - Added flow (f4c03c74-dee9-42cb-8f72-cfd84084bdbe) with jobs: ('9eb1ebd0-12c5-4ef7-9b3b-c4adf132403d',)


['74']